## Import the libraries


In [1]:
!pip install -U "bitsandbytes>=0.46.0" "accelerate>=1.0.0" "peft>=0.12.0" "trl>=0.12.0" "datasets>=3.0.0"
!pip install -U "setuptools<70"
!pip install -U cvlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 927.2/927.2 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 35.7 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 23.0.1
    Uninstalling pyarrow-23.0.1:
      Successfully uninstalled pyarrow-23.0.1
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Un

In [2]:
import os
from collections import Counter
import json
import re

import torch
from datasets import Dataset, DatasetDict, load_dataset
from huggingface_hub import snapshot_download
from peft import LoraConfig, prepare_model_for_kbit_training

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# Transformer Reinforcement Learning Library
from trl import SFTConfig, SFTTrainer

## Check the GPU


In [3]:
if not torch.cuda.is_available():
    raise RuntimeError("A GPU is required for this project")

device = torch.device("cuda")
print(torch.cuda.get_device_name(0))
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


Tesla T4
Memory: 14.6 GB


## Create the project folders


In [4]:
WORKSPACE = "/tmp/tensortonic"
DATA_DIR = os.path.join(WORKSPACE, "data", "medquad")
MODEL_DIR = os.path.join(WORKSPACE, "models")
MODEL_CACHE = os.path.join(MODEL_DIR, "huggingface")
DATASET_CACHE = os.path.join(WORKSPACE, "cache", "datasets")

os.makedirs(MODEL_CACHE, exist_ok=True)
os.makedirs(DATASET_CACHE, exist_ok=True)

print(WORKSPACE)
print(MODEL_DIR)
print(MODEL_CACHE)
print(DATASET_CACHE)


/tmp/tensortonic
/tmp/tensortonic/models
/tmp/tensortonic/models/huggingface
/tmp/tensortonic/cache/datasets


## Download the base model


In [5]:
MODEL_ID = "Qwen/Qwen3-4B"
print(f"Downloading {MODEL_ID} to {MODEL_CACHE}", flush=True)
local_model_path = snapshot_download(MODEL_ID, cache_dir=MODEL_CACHE)
print(f"Model files ready at {local_model_path}", flush=True)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Model files ready at /tmp/tensortonic/models/huggingface/models--Qwen--Qwen3-4B/snapshots/1cfa9a7208912126459214e8b04321603b3df60c


## Configure 4-bit quantization


### ¿Por qué hay que hacer esto? (Explicación conceptual)

Esta configuración es la base de una técnica fundamental en el fine-tuning actual llamada **QLoRA** (*Quantized Low-Rank Adaptation*).

Los Modelos de Lenguaje Grandes (LLMs) tienen miles de millones de parámetros. Si intentas cargar un modelo de 7 mil millones de parámetros (7B) con precisión normal (16 bits / FP16), solo el modelo ocupará unos **14 GB - 16 GB de VRAM** en la GPU. Si a eso le sumas los gradientes, optimizadores y activaciones del entrenamiento, fácilmente superarías los 24 GB o más, provocando un error de memoria (*Out of Memory* o *CUDA OOM*).

El objetivo de esta celda es **reducir drásticamente el uso de memoria RAM de tu GPU** para que el modelo quepa y pueda entrenarse en GPUs comerciales o gratuitas (como las de Google Colab de 15 GB).

---

### ¿Qué hace cada parámetro?

1. **`load_in_4bit=True`**:
   * **Qué hace:** Comprime los pesos del modelo base de 16 bits a solo **4 bits**.
   * **Por qué:** Reduce el tamaño en memoria del modelo a una cuarta parte (un modelo de 7B pasa de ocupar ~14-16 GB a solo ~4-5 GB).

2. **`bnb_4bit_quant_type="nf4"`** (*NormalFloat 4*):
   * **Qué hace:** Define el algoritmo de compresión. En lugar de una cuantización lineal simple (`fp4`), utiliza **NF4**.
   * **Por qué:** Los pesos de las redes neuronales siguen naturalmente una distribución normal (campana de Gauss). NF4 está diseñado matemáticamente para adaptarse a esta distribución, logrando mucha menor pérdida de calidad/precisión que los formatos de 4 bits estándar.

3. **`bnb_4bit_compute_dtype=torch.bfloat16`**:
   * **Qué hace:** Aunque los pesos estén almacenados en 4 bits, los cálculos matemáticos (multiplicaciones de matrices) se descomprimen momentáneamente a **bfloat16** durante las operaciones.
   * **Por qué:** Realizar cálculos directamente en 4 bits degradaría gravemente el rendimiento del modelo. Hacer los cálculos en `bfloat16` (o `float16`) mantiene la estabilidad numérica y la velocidad de la GPU sin necesidad de mantener todo el modelo en 16 bits en reposo.

4. **`bnb_4bit_use_double_quant=True`** (*Doble cuantización*):
   * **Qué hace:** Cuantiza también las constantes de cuantización (los factores de escala que se usan para comprimir a 4 bits).
   * **Por qué:** Ahorra aproximadamente **0.37 bits por parámetro** adicionales en memoria (alrededor de 300 MB - 500 MB en un modelo de 7B). Parece poco, pero en GPUs con memoria ajustada puede marcar la diferencia entre que el entrenamiento funcione o falle por falta de memoria.

### 3. ¿Qué son las constantes de cuantización?

Para comprimir un grupo de números a solo 16 valores posibles sin perder el orden de magnitud, no puedes usar la misma escala fija para todo el modelo (porque una capa puede tener números entre $$-0.01$$ y $$0.01$$, y otra capa entre $$-2.0$$ y $$2.0$$).

Entonces, lo que hace el algoritmo es dividir los pesos en pequeños bloques (normalmente de 64 números). Para cada bloque calcula un **factor de escala** (la **constante de cuantización**, normalmente llamada $$c$$):

$$\text{Peso cuantizado} = \text{redondear}\left(\frac{\text{Peso original}}{c}\right)$$

* El peso se guarda en **4 bits**.
* Pero necesitas guardar esa constante $$c$$ en algún sitio para que la GPU sepa cómo "deshacer" la división y recuperar el número cuando vaya a calcular.
* El problema: esa constante $$c$$ originalmente se guardaba en **FP32 o FP16**. Si tienes una constante por cada 64 pesos, todas esas constantes juntas terminan ocupando bastante memoria (unos 0.5 bits extra por cada parámetro).

**Aquí entra la Doble Cuantización (`bnb_4bit_use_double_quant=True`):**
Consiste en **cuantizar las propias constantes de cuantización** (por ejemplo, pasarlas de 32 bits a 8 bits). Es comprimir las claves de descompresión, ahorrando memoria extra sin perder calidad.

---

### 4. ¿Son estas variables las que siempre se usan para cuantizar LLMs?

**Para fine-tuning con QLoRA en Hugging Face, SÍ.** Esta configuración exacta:
```python
bnb_4bit_quant_type="nf4",
bnb_4bit_compute_dtype=torch.bfloat16,
bnb_4bit_use_double_quant=True
```
es el **estándar de la industria** desde que se publicó el paper de QLoRA en 2023. Se considera la mejor combinación entre ahorro de memoria y preservación del rendimiento del modelo.

**Sin embargo, fuera de QLoRA (para inferencia/uso diario), existen otros formatos:**
Si tu objetivo **no es entrenar**, sino solo **ejecutar** el modelo para chatear o hacer predicciones:
* **GGUF (llama.cpp):** Muy popular para ejecutar LLMs en local usando CPUs o GPUs de consumo/Macs. Utiliza sus propios esquemas (Q4_K_M, Q5_K_M, etc.).
* **AWQ / GPTQ:** Métodos de cuantización muy rápidos a nivel de cálculo para servidores en producción que solo hacen inferencia (por ejemplo, usando frameworks como vLLM o TGI).

En resumen:
* **Para Fine-Tuning (entrenamiento ligero):** `BitsAndBytesConfig` con NF4 + bfloat16 + double quant es la reina indiscutible.
* **Para Inferencia pura:** GGUF, AWQ o GPTQ suelen ser más rápidos en velocidad de generación por token.

In [6]:
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    # TODO 1: configure NF4, bfloat16 compute, and double quantization.
    bnb_4bit_quant_type="nf4", #nf4 is design to adapt normal distribution gettin less losing that standard 4 bits
    bnb_4bit_compute_dtype=torch.bfloat16, # calculating matrix on bfloat16 is more optimal
    bnb_4bit_use_double_quant=True,
)
print(quantization)

BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "bfloat16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}



## Load the tokenizer and model


Tokenizer and pad strategy

loading and quantization on GPU

Esta es la celda **crucial** donde todo lo que descargaste en el disco y la configuración matemática que definiste antes finalmente **cobran vida dentro de tu GPU**.

Vamos a desglosarla línea por línea para entender qué hace y por qué es estrictamente necesario:

---

### 1. El Tokenizador y el truco del `pad_token`

```python
tokenizer = AutoTokenizer.from_pretrained(local_model_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
```

* **`AutoTokenizer.from_pretrained(...)`:** Carga los archivos del vocabulario (`merges.txt`, `tokenizer.json`) para convertir texto humano en listas de números (tokens).
* **¿Qué es `pad_token` y por qué se iguala a `eos_token`?**
  * Cuando entrenas un LLM, no le pasas una sola frase; le pasas grupos de frases en lotes (*batches*), por ejemplo de 4 en 4.
  * Para que la GPU pueda multiplicar matrices en paralelo, **todas las frases del lote deben tener exactamente la misma longitud**.
  * Si la Frase 1 tiene 10 palabras y la Frase 2 tiene 6 palabras, a la Frase 2 le faltan 4 huecos. Hay que "rellenarla" (*padding*) con un token especial (`pad_token`).
  * **El problema:** La mayoría de modelos generativos (como Llama o Qwen) están pensados originalmente solo para inferencia texto a texto, por lo que **no traen un `pad_token` definido de fábrica**.
  * **La solución:** Si es `None`, le decimos al tokenizador que use el `eos_token` (*End of Sequence*, el token de fin de texto) como relleno provisional para que no dé un error al armar los lotes de entrenamiento.

---

### 2. Carga y Cuantización real en la GPU

```python
base_model = AutoModelForCausalLM.from_pretrained(
    local_model_path,
    quantization_config=quantization,
    device_map={"": 0},
    torch_dtype=torch.bfloat16,
)
```

Aquí ocurre lo que vimos antes:
* **`local_model_path`:** Lee los archivos `.safetensors` desde la carpeta `snapshots/` (que apuntan a los `blobs` de ~3.8 GB).
* **`quantization_config=quantization`:** Activa la receta que configuraste en la celda anterior (`BitsAndBytesConfig`). A medida que lee los 7.5 GB de pesos desde el disco, **los comprime a 4 bits (NF4) y los almacena en la VRAM de la GPU ocupando solo ~2.5 GB**.
* **`device_map={"": 0}`:** Le dice a PyTorch: *"Coloca todo el modelo directamente en la GPU número 0"* (tu tarjeta gráfica principal). Si no pones esto, intentaría cargarlo en la memoria RAM de la CPU.
* **`torch_dtype=torch.bfloat16`:** Define la precisión para aquellas capas que no se cuantizan (o para las operaciones intermedias). *(El aviso amarillo que te salió solo indica que en versiones nuevas de Hugging Face prefieren que escribas simplemente `dtype=torch.bfloat16`)*.

---

### 3. Apagar la caché (`use_cache = False`)

```python
base_model.config.use_cache = False
```

* **¿Qué es la caché (*KV Cache*)?**
  Cuando estás chateando con ChatGPT y genera texto palabra por palabra, el modelo guarda en memoria lo que ya calculó de las palabras anteriores para no recalcularlo en cada token nuevo. Esto hace que generar texto sea muy rápido.
* **¿Por qué la apagamos al entrenar?**
  Durante el **fine-tuning**, le pasas al modelo el texto completo de golpe (pregunta y respuesta juntas) y calculas gradientes hacia atrás (*backpropagation*).
  * La caché no sirve de nada durante el entrenamiento.
  * Además, la caché es incompatible con una técnica de ahorro de memoria que se usa más adelante llamada *Gradient Checkpointing*. Si dejas `use_cache = True`, el entrenamiento fallará con un error.

---

### 4. Preparar el modelo para entrenar en 4 bits

```python
base_model = prepare_model_for_kbit_training(base_model)
```

Esta función viene de la librería **PEFT** (*Parameter-Efficient Fine-Tuning*) de Hugging Face y es indispensable para QLoRA. Hace tres ajustes internos bajo el capó:

1. **Congela todos los pesos base del modelo:** Nadie va a tocar ni modificar los 4.000 millones de parámetros cuantizados del modelo original.
2. **Casting de capas críticas a FP32 (precisión completa):** Hay ciertas capas muy sensibles en una red neuronal (como las capas de normalización, *LayerNorm*) donde redondear a 4 bits provocaría inestabilidad matemática y haría que la pérdida (*loss*) se fuera a `NaN` (infinito). Esta función se asegura de que esas capas se mantengan en máxima precisión.
3. **Prepara las entradas para el cálculo de gradientes:** Permite que las activaciones retengan gradientes para poder entrenar los futuros adaptadores LoRA que inyectarás en la siguiente celda.

In [7]:
tokenizer = AutoTokenizer.from_pretrained(local_model_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# LOAD AND FREEZE QUANTIZED BASE LLM ON GPU 4 APPLY ADAPTERS QLORA
base_model = AutoModelForCausalLM.from_pretrained(
    local_model_path, # read blobs from llm located on ssd
    quantization_config=quantization, # applied previous quantization config
    device_map={"": 0}, #set all quantized model on number 0 gpu instead of ram cpu
    torch_dtype=torch.bfloat16, #define precision 4 intermediate layers
)
base_model.config.use_cache = False # cache on trainin is useless, it must be used on inference
base_model = prepare_model_for_kbit_training(base_model) # 1. freeze llm base wights knowledge. 2. keep critic layers on fp32 3. prepare 4 compute gradients on trainin
print(f"Loaded {MODEL_ID}")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loaded Qwen/Qwen3-4B


### ¿En qué estado quedó el modelo ahora mismo?
En este punto:
* Tu GPU tiene cargado el modelo Qwen3-4B comprimido a 4 bits en solo ~2.5 GB de VRAM.
* El modelo base está completamente **congelado** (bloqueado para no sobreescribir sus conocimientos).
* Está listo para que en el siguiente paso le agregues los pequeños adaptadores entrenables de **LoRA** (los adapters), que serán las únicas piezas que realmente aprenderán durante el fine-tuning.

## Download MedQuAD dataset 4 trainin


En la celda anterior preparaste el "cerebro" (el modelo y su tokenizador). En esta celda estás preparando **el "libro de estudio" con el que vas a especializar ese cerebro**: el conjunto de datos (*dataset*).

El objetivo general es descargar un conjunto de datos médicos en formato `.csv`, convertirlo a `.jsonl` y dejarlo guardado localmente listo para entrenar.

Vamos a analizar línea por línea qué hace y por qué se hace de esta manera:

---

### 1. Definir el origen de los datos

```python
DATASET_ID = "Hmehdi515/MedQuad"
DATASET_FILES = {
    "train": "https://huggingface.co/datasets/Hmehdi515/MedQuad/resolve/main/medquad_train.csv",
    "validation": "https://huggingface.co/datasets/Hmehdi515/MedQuad/resolve/main/medquad_val.csv",
}
```

* **¿Qué es MedQuAD?** 
  Significa *Medical Question Answering Dataset*. Es un conjunto de datos muy famoso en medicina que contiene miles de parejas de **preguntas y respuestas sobre salud, enfermedades y medicamentos**, extraídas de fuentes médicas oficiales (como los Institutos Nacionales de Salud de EE.UU., NIH).
* **`DATASET_FILES`**: En lugar de descargar todo el repositorio a ciegas, le dices exactamente las URLs de descarga directa de dos particiones:
  * **`train`**: El conjunto con el que el modelo va a aprender (16.078 ejemplos).
  * **`validation`**: Un pequeño examen final de 164 ejemplos que el modelo no verá durante el entrenamiento, usado para medir si realmente está aprendiendo o memorizando.

---

### 2. Descarga y estructuración en memoria

```python
print(f"Downloading {DATASET_ID}", flush=True)
source = load_dataset("csv", data_files=DATASET_FILES, cache_dir=DATASET_CACHE)
```

* **`load_dataset("csv", ...)`**: Viene de la librería `datasets` de Hugging Face.
  * Lee los archivos CSV remotos a través de internet.
  * Los convierte internamente a un formato de memoria ultrarrápido basado en **Apache Arrow** (por eso en la salida viste `Creating json from Arrow format`).
  * Guarda una copia en caché en `cache_dir` para que si vuelves a ejecutar la celda no tenga que descargarlos otra vez de internet.

---

### 3. Convertir y Guardar en formato `.jsonl`

```python
os.makedirs(DATA_DIR, exist_ok=True)
raw_train_path = os.path.join(DATA_DIR, "raw_train.jsonl")
raw_validation_path = os.path.join(DATA_DIR, "raw_validation.jsonl")
source["train"].to_json(raw_train_path)
source["validation"].to_json(raw_validation_path)
```

* **`os.makedirs(DATA_DIR, exist_ok=True)`**: Crea la carpeta local (en `/tmp/tensortonic/data/medquad/`) si no existía ya.
* **¿Por qué convertir de `.csv` a `.jsonl` (`.to_json`)?**
  Esta es una práctica estándar en el entrenamiento de LLMs:
  * Los archivos CSV suelen dar problemas con textos largos porque las respuestas médicas tienen comas, comillas, saltos de línea y caracteres especiales que pueden romper las columnas del CSV.
  * **JSONL (JSON Lines)** guarda **cada ejemplo en una sola línea de texto independiente** como un diccionario JSON:
    ```json
    {"sample": 1, "question": "¿Cuáles son los síntomas de la gripe?", "answer": "Fiebre, tos, dolor..."}
    {"sample": 2, "question": "¿Qué causa la diabetes?", "answer": "La falta de insulina..."}
    ```
  * Esto permite leer los datos línea por línea (*streaming*) sin tener que cargar gigabytes de texto en la memoria RAM a la vez.

---

### 4. ¿Qué te muestra la salida final?

Al imprimir `source`, la consola te muestra el objeto `DatasetDict`:

```text
DatasetDict({
    train: Dataset({
        features: ['sample', 'question', 'answer'],
        num_rows: 16078
    })
    validation: Dataset({
        features: ['sample', 'question', 'answer'],
        num_rows: 164
    })
})
```

Esto te confirma:
1. **`num_rows`**: Tienes **16.078 ejemplos** de entrenamiento y **164** de validación.
2. **`features` (columnas)**: Cada ejemplo tiene 3 campos:
   * `'sample'`: Un número identificador.
   * `'question'`: La pregunta médica que el usuario haría al modelo.
   * `'answer'`: La respuesta médica profesional correcta que quieres que el modelo aprenda a dar.

In [8]:
DATASET_ID = "Hmehdi515/MedQuad"
DATASET_FILES = {
    "train": "https://huggingface.co/datasets/Hmehdi515/MedQuad/resolve/main/medquad_train.csv",
    "validation": "https://huggingface.co/datasets/Hmehdi515/MedQuad/resolve/main/medquad_val.csv",
}

print(f"Downloading {DATASET_ID}", flush=True)
#download remote csv 4 trainin and save em to cache
source = load_dataset("csv", data_files=DATASET_FILES, cache_dir=DATASET_CACHE)

#convert csv to jsonl and save it
os.makedirs(DATA_DIR, exist_ok=True)
raw_train_path = os.path.join(DATA_DIR, "raw_train.jsonl")
raw_validation_path = os.path.join(DATA_DIR, "raw_validation.jsonl")
source["train"].to_json(raw_train_path) # llm trainin standard practice. convert to jsonl
source["validation"].to_json(raw_validation_path)

print(f"Saved {len(source['train'])} rows to {raw_train_path}", flush=True)
print(f"Saved {len(source['validation'])} rows to {raw_validation_path}", flush=True)
print(source)


medquad_train.csv: reconstructing file:   0%|          |  0.00B / 22.0MB            

medquad_train.csv: downloading bytes:           |  0.00B            

medquad_val.csv:   0%|          | 0.00/199k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Creating json from Arrow format:   0%|          | 0/17 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Saved 16078 rows to /tmp/tensortonic/data/medquad/raw_train.jsonl
Saved 164 rows to /tmp/tensortonic/data/medquad/raw_validation.jsonl
DatasetDict({
    train: Dataset({
        features: ['sample', 'question', 'answer'],
        num_rows: 16078
    })
    validation: Dataset({
        features: ['sample', 'question', 'answer'],
        num_rows: 164
    })
})



### ¿Cuál es el siguiente paso lógico en el notebook?
Ahora mismo tienes el texto plano en disco (`question` y `answer`), pero los LLMs no leen texto crudo. 

En la siguiente celda seguramente harás el **preprocesamiento**: unirás la pregunta y la respuesta dentro de una plantilla de chat o instrucción (por ejemplo: `### Instrucción: {question} \n ### Respuesta: {answer}`) y usarás el `tokenizer` que cargaste antes para transformar ese texto en números antes de pasárselo a la GPU.

## Inspect the dataset


Inspect the available columns and calculate the median answer length for **the training split.**

Check: The available dataset columns, a non-empty list of answer lengths, and one median value.

Read column_names from the training split. Sort the word counts. For an even number of answers, average the two middle values; otherwise select the middle value.

Inspeccionar las columnas garantiza que el dataset tiene los datos correctos para el prompt, y calcular la mediana de palabras te permite ajustar el tamaño de contexto del modelo con precisión milimétrica, maximizando la velocidad de entrenamiento y evitando que la GPU se quede sin memoria.

Si calculas el promedio, esos pocos textos gigantes inflarán el resultado artificialmente y te darán un número engañoso (por ejemplo, un promedio de 450 palabras).
La mediana, en cambio, es inmune a esos valores extremos y te da la verdadera longitud de una respuesta típica.

**Sobre la razón por la que un Ingeniero de ML siempre hace esto antes de tocar una sola línea de entrenamiento**.

### ¿Por qué un Ingeniero de ML hace esto? ¿Para qué sirve la mediana?

A primera vista parece un simple ejercicio escolar de estadística ("contar palabras y sacar la mediana"), pero en el mundo de los LLMs tiene un impacto técnico y económico directo sobre **la GPU, la velocidad y la memoria**:

#### 1. El dilema del `max_seq_length` (La longitud máxima del modelo)
Cuando entrenas un LLM, debes definir un hiperparámetro obligatorio: **cuántos tokens como máximo vas a procesar por texto** (por ejemplo, `max_seq_length = 512` o `max_seq_length = 2048`).
* Si lo pones a ciegas muy alto (ej. `4096`), la GPU colapsará por falta de memoria (CUDA OOM), o el entrenamiento irá 5 veces más lento de lo necesario.
* Si lo pones muy bajo (ej. `128`), recortarás las respuestas médicas por la mitad y el modelo aprenderá a dar respuestas incompletas o cortadas.

Saber la longitud típica de las respuestas te dice exactamente qué valor de `max_seq_length` necesitas configurar más adelante.

#### 2. ¿Por qué la MEDIANA y no el PROMEDIO (Media)?
En procesamiento de lenguaje natural (NLP), la longitud de los textos sigue una distribución con una **cola muy larga** (*skewed distribution*):
* La inmensa mayoría de las respuestas médicas tienen, digamos, entre **50 y 120 palabras**.
* Pero suele haber 4 o 5 ejemplos que son manuales clínicos completos copiados de golpe de **8.000 palabras** (outliers o anomalías).

Si calculas el **promedio**, esos pocos textos gigantes inflarán el resultado artificialmente y te darán un número engañoso (por ejemplo, un promedio de 450 palabras). 
La **mediana**, en cambio, es inmune a esos valores extremos y te da la **verdadera longitud de una respuesta típica**.

#### 3. El coste cuadrático de la atención de los Transformers
La atención en los Transformers tiene una complejidad computacional de **$$O(N^2)$$**, donde $$N$$ es el número de tokens:
* Duplicar la longitud de la secuencia no duplica el uso de memoria de atención: **lo cuadruplica ($$2^2 = 4$$)**.
* Si por no inspeccionar tu dataset configuras un tamaño de secuencia innecesariamente largo solo por miedo a cortar textos, estarás desperdiciando el 80% de tu VRAM calculando atención sobre espacios vacíos (*padding tokens*).

### En resumen
Inspeccionar las columnas garantiza que el dataset tiene los datos correctos para el prompt, y calcular la mediana de palabras te permite **ajustar el tamaño de contexto del modelo con precisión milimétrica**, maximizando la velocidad de entrenamiento y evitando que la GPU se quede sin memoria.

In [9]:
# TODO 2: inspect the columns and calculate the median answer length in words.
columns = source["train"].column_names
answer_lengths = [len(answer.split()) for answer in source["train"]["answer"]]
sorted_lenghts = sorted(answer_lengths)
n = len(sorted_lenghts)
# we need to know how many tokens gonna be proccesed in thre train phase
if n % 2 == 1:
    median_answer_words = sorted_lenghts[n // 2]
else:
    median_answer_words = (sorted_lenghts[(n // 2) - 1] + sorted_lenghts[n // 2]) / 2.0

assert {"question", "answer"}.issubset(columns), "Check the dataset columns"
assert answer_lengths, "Calculate the answer lengths"
assert median_answer_words is not None, "Calculate the median answer length"

print({
    "columns": columns,
    "training_rows": len(source["train"]),
    "median_answer_words": median_answer_words,
})
print(source["train"][0])


{'columns': ['sample', 'question', 'answer'], 'training_rows': 16078, 'median_answer_words': 139.0}
{'sample': 12536, 'question': 'What are the symptoms of Male Breast Cancer ?', 'answer': 'Men with breast cancer usually have lumps that can be felt.Lumps and other signs may be caused by male breast cancer or by other conditions. Check with your doctor if you notice a change in your breasts.'}


## Prepare a clean training split


Complete the filtering condition used to build the training and evaluation splits.

Check: 96 training rows and 8 evaluation rows, with every answer between 20 and 160 words.

Use the cleaned question and answer_words values when setting keep_example.

En esta celda se est'a haciendo lo que en la industria se conoce como **Data Curation & Filtering**, además de crear un subconjunto controlado para un entrenamiento rápido.

---

```python
        # TODO 3: keep non-empty questions with answers between 20 and 160 words.
        keep_example = len(question) > 0 and (20 <= answer_words <= 160)
```

Al colocar esta línea:
1. `len(question) > 0`: Asegura que la pregunta no sea un texto vacío o puros espacios en blanco.
2. `20 <= answer_words <= 160`: Se asegura de que la respuesta tenga al menos 20 palabras y como máximo 160.

---

### ¿Por qué se hace esto? Explicación detallada

#### 1. ¿Por qué filtrar entre 20 y 160 palabras? ("Calidad sobre Cantidad")
En el fine-tuning de LLMs existe la regla de oro: **"Garbage in, garbage out"** (si metes basura, obtienes basura).

* **Descartar respuestas menores a 20 palabras (`< 20`):**
  En medicina, una respuesta ultracorta suele ser inútil o peligrosa (ejemplo: *"Consulte a su médico"*, *"Sí"*, *"No se sabe"*). Si entrenas al modelo con textos tan breves, aprenderá a ser perezoso y evasivo.
* **Descartar respuestas mayores a 160 palabras (`> 160`):**
  Basado en la celda anterior (donde calculaste la mediana), descubriste cuál es la longitud estándar de una buena respuesta médica. Los textos con más de 160 palabras suelen ser listas interminables copiadas de prospectos médicos o historiales clínicos. Además, limitar a 160 palabras te garantiza que **ningún ejemplo se cortará por falta de contexto en la GPU** y que el entrenamiento será sumamente rápido y ligero en VRAM.

#### 2. ¿Por qué solo 96 ejemplos de entrenamiento y 8 de evaluación?
Si el dataset original tenía 16.078 ejemplos, ¿por qué quedarse solo con 96?
* **Propósito de Laboratorio/Práctica:** En un entorno de aprendizaje o prueba de concepto, entrenar 16.000 filas con un LLM tomaría varias horas. Con 96 filas puedes completar el ciclo entero en **1 o 2 minutos** y comprobar que todo tu pipeline de QLoRA funciona sin gastar cómputo innecesario.
* **El poder de LoRA / Instruction Tuning:** Curiosamente, para enseñar a un modelo a *adoptar un estilo o formato* (en este caso, responder como médico conciso), **unas pocas decenas o cientos de ejemplos de altísima calidad suelen ser suficientes** (este es el famoso principio demostrado por Meta en su paper *LIMA: Less Is More for Alignment*).

#### 3. El `shuffle(seed=42)`
```python
"train": select_examples("train", 96).shuffle(seed=42)
```
* Si no barajas los datos, los 96 ejemplos podrían venir en el orden original del archivo (donde a menudo las primeras 100 preguntas son todas sobre la misma enfermedad, ej. "Cáncer de mama", porque se extrajeron por orden alfabético).
* Barajar con una semilla fija (`seed=42`) garantiza que el modelo vea variedad de temas y que el experimento sea **100% reproducible** cada vez que lo ejecutes.

#### 4. Guardar en dos formatos distintos
```python
dataset["train"].to_json(...)        # Guarda en JSON Lines (.jsonl)
dataset.save_to_disk(..., "arrow")   # Guarda en formato nativo Apache Arrow
```
* **`.jsonl`:** Es para ti, para humanos y para interoperabilidad. Puedes abrirlo con cualquier editor de texto o script para inspeccionar qué se guardó.
* **`arrow`:** Es el formato binario nativo de Hugging Face. Permite que la librería lea los datos mediante mapeo de memoria (*memory mapping*), cargando los ejemplos en la GPU a velocidad de vértigo sin pasar por el interprete lento de Python.

#### 5. El `SYSTEM_PROMPT`
```python
SYSTEM_PROMPT = (
    "Answer the medical question clearly and concisely. When a question "
    "requires personal medical advice, recommend consulting a qualified clinician."
)
```
Aquí estás definiendo la **personalidad y la directiva de seguridad del modelo**:
* Le dices el rol: Responder preguntas médicas de forma clara y concisa.
* Le pones una **barrera de contención ética/legal (*guardrail*)**: si el usuario pide una consulta personal, debe recordarle que acuda a un médico colegiado. 
* En la siguiente celda, este `SYSTEM_PROMPT` se concatenará con la `question` y la `answer` para crear el prompt final que verá el tokenizador.

In [10]:
def select_examples(split, limit):
    rows = []
    for example in source[split]:
        question = str(example["question"]).strip()
        answer = str(example["answer"]).strip()
        answer_words = len(answer.split())

        # TODO 3: keep non-empty questions with answers between 20 and 160 words.
        keep_example = len(question) > 0 and (20 <= answer_words <= 160)
        if keep_example:
            rows.append({"question": question, "answer": answer})
        if len(rows) == limit:
            break
    return Dataset.from_list(rows)

print("Creating a small training set and a separate evaluation set", flush=True)
dataset = DatasetDict({
    "train": select_examples("train", 96).shuffle(seed=42),
    "evaluation": select_examples("validation", 8),
})

print(f"Writing dataset files to {DATA_DIR}", flush=True)
dataset["train"].to_json(os.path.join(DATA_DIR, "train.jsonl"))
dataset["evaluation"].to_json(os.path.join(DATA_DIR, "evaluation.jsonl"))
dataset.save_to_disk(os.path.join(DATA_DIR, "arrow"))

SYSTEM_PROMPT = (
    "Answer the medical question clearly and concisely. When a question "
    "requires personal medical advice, recommend consulting a qualified clinician."
)
print(dataset)
print(dataset["train"][0])


Creating a small training set and a separate evaluation set
Writing dataset files to /tmp/tensortonic/data/medquad


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Saving the dataset (0/1 shards):   0%|          | 0/96 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/8 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 96
    })
    evaluation: Dataset({
        features: ['question', 'answer'],
        num_rows: 8
    })
})
{'question': "What is (are) Crohn's Disease ?", 'answer': "Crohn's disease is a chronic, or long lasting, disease that causes inflammationirritation or swellingin the gastrointestinal (GI) tract. Most commonly, Crohn's affects the small intestine and the beginning of the large intestine. However, the disease can affect any part of the GI tract, from the mouth to the anus.\n                \nCrohn's disease is a chronic inflammatory disease of the GI tract, called inflammatory bowel disease (IBD). Ulcerative colitis and microscopic colitis are the other common IBDs. More information is provided in the NIDDK health topics, Ulcerative Colitis and Microscopic Colitis: Collagenous Colitis and Lymphocytic Colitis.\n                \nCrohn's disease most often begins gradually and can become w

## Build the generation prompt


Create the messages for a medical question and render the prompt used for generation.

Check: System and user messages rendered with a generation prompt and returned as PyTorch tensors.

Call apply_chat_template with add_generation_prompt, return_tensors, and return_dict enabled. Disable thinking.

In [11]:
def generate_answer(model, question):
    # TODO 4: create the system and user messages, then render the generation prompt.
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    # tokenizer for convert text to tokens for the model
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt", #convert string to pytortch tensor for the GPU works
        return_dict=True,
        enable_thinking=False,  # O thinking=False según la versión de Qwen
    )

    assert messages, "Add the prompt messages"
    assert inputs is not None, "Render the prompt with the tokenizer"
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    prompt_length = inputs["input_ids"].shape[-1]
    # turn off gradient compute, i want just inference mode. Keep compute and gain speed with just forward pass
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=160, # no more that median words 160
            do_sample=False,# not sampling, deterministic outputs for medical outputs
            pad_token_id=tokenizer.pad_token_id,
        )
    # let clean text response, jsut new tokens without original question    
    return tokenizer.decode(output[0][prompt_length:], skip_special_tokens=True).strip()


## Implement the evaluation metric


Implement the evaluation metric
TODO 5
Implement token F1 so the base and fine-tuned models can be compared with the same metric.

Check: Lowercase word tokens and an overlap count that handles repeated words correctly.

Use re.findall for word tokens. Counter intersection gives the repeated-token overlap.

¿Por qué se usa F1-Score para evaluar texto médico y qué significa cada parte?
En tareas como clasificación o visión por computador, la métrica es simple: "acierto o fallo" (Accuracy). Pero en respuestas de texto libre generadas por un LLM, la precisión exacta al 100% no existe: el modelo puede responder lo mismo que el médico, pero usando sinónimos o cambiando el orden de las frases.

El Token F1 mide el grado de coincidencia entre las palabras de la Predicción del modelo y la Referencia (la respuesta médica real).

Por ello, el estándar de referencia (establecido por el benchmark SQuAD de Rajpurkar et al., Stanford) es el Token F1-Score:

3. Precisión, Recall y F1: La armonía perfecta
Precisión (
Precision
Precision): De todas las palabras que dijo el modelo, ¿cuántas eran relevantes?
Precision
=
palabras coincidentes
palabras que dijo el modelo
Precision= 
palabras que dijo el modelo
palabras coincidentes
​
 
(Castiga al modelo si es charlatán y mete "paja" o información irrelevante).

Exhaustividad (
Recall
Recall): De todas las palabras médicas correctas, ¿cuántas logró rescatar el modelo?
Recall
=
palabras coincidentes
palabras de la respuesta real
Recall= 
palabras de la respuesta real
palabras coincidentes
​
 
(Castiga al modelo si fue demasiado tímido y dejó la respuesta incompleta).

F
1
-Score
F 
1
​
 -Score: Es la media armónica entre ambos. Va de 0.0 (ninguna coincidencia) a 1.0 (coincidencia de vocabulario idéntica):
F
1
=
2
×
Precision
×
Recall
Precision
+
Recall
F 
1
​
 =2× 
Precision+Recall
Precision×Recall
​
 

Con esta métrica podrás medir objetivamente con un número matemático cuánto mejoró el modelo una vez que le apliques el fine-tuning con LoRA frente a cómo respondía al principio.

In [12]:
def answer_f1(prediction, reference):
    # TODO 5: tokenize both answers and count their overlapping words.
    # get words with re.findall() and \w+ for just letters and numbers
    predicted_tokens = re.findall(r"\w+", prediction.lower())
    reference_tokens = re.findall(r"\w+", reference.lower())
    
    if not predicted_tokens or not reference_tokens:
        return float(predicted_tokens == reference_tokens)

    # compute words frequency and intersection
    pred_counts = Counter(predicted_tokens)
    ref_counts = Counter(reference_tokens)
    overlap = sum((pred_counts & ref_counts).values())
    
    if overlap == 0:
        return 0.0
    precision = overlap / len(predicted_tokens)
    recall = overlap / len(reference_tokens)
    return 2 * precision * recall / (precision + recall)


## Measure the base model


en la ingeniería de Machine Learning se llama establecer la línea base (Baseline).

Consiste en someter al modelo base (tal cual lo descargaste, sin haber aprendido nada todavía de tu dataset) al "examen inicial" para tener una puntuación de referencia con la que comparar después del fine-tuning.

In [13]:
def evaluate_model(model, rows):
    model.eval() #tell to pytorch deactivate trainin layers lookin for stable and deterministic predictions while eval
    results = []
    for row in rows:
        prediction = generate_answer(model, row["question"])
        score = answer_f1(prediction, row["answer"])
        results.append({
            "question": row["question"],
            "reference": row["answer"],
            "prediction": prediction,
            "token_f1": round(score * 100, 2),
        })
    return {
        "token_f1": round(sum(row["token_f1"] for row in results) / len(results), 2),
        "examples": len(results),
        "results": results,
    }

baseline = evaluate_model(base_model, dataset["evaluation"])
print(json.dumps({
    "token_f1": baseline["token_f1"],
    "examples": baseline["examples"],
}, indent=2))


{
  "token_f1": 25.96,
  "examples": 8
}


línea base de 25.96% de Token F1 es un punto de partida muy realista para un modelo base de 0.5B a 1.5B parámetros que aún no conoce el estilo ni la concisión de este dataset. Ese será el número a batir después del entrenamiento.

## Format examples as conversations


fase de preparación de datos para el entrenamiento (Supervised Fine-Tuning o SFT).

1. ¿Por qué incluimos el rol "assistant" aquí?
Porque en el aprendizaje supervisado (Supervised Fine-Tuning), el modelo aprende por imitación. Necesita ver la secuencia completa:

Las instrucciones de cómo comportarse (system).
Lo que pregunta el paciente (user).
La respuesta correcta ideal redactada por los médicos del dataset (assistant).

Format examples as conversations
TODO 6
Build the system, user, and assistant messages, then render them with the tokenizer chat template.

Check: Each row contains one non-empty text field with the complete conversation.

Use SYSTEM_PROMPT, example["question"], and example["answer"]. Keep tokenize and enable_thinking disabled.

In [14]:
def format_example(example):
    # TODO 6: add system, user, and assistant messages in that order.
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["question"]},
        {"role": "assistant", "content": example["answer"]},
    ]


    # TODO 6: render the messages as text with the tokenizer chat template.
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        enable_thinking=False,
    )
    return {"text": text}

train = dataset["train"].map(format_example)
assert train[0]["text"].strip(), "The formatted training text is empty"
print(train[0]["text"])


Map:   0%|          | 0/96 [00:00<?, ? examples/s]

<|im_start|>system
Answer the medical question clearly and concisely. When a question requires personal medical advice, recommend consulting a qualified clinician.<|im_end|>
<|im_start|>user
What is (are) Crohn's Disease ?<|im_end|>
<|im_start|>assistant
<think>

</think>

Crohn's disease is a chronic, or long lasting, disease that causes inflammationirritation or swellingin the gastrointestinal (GI) tract. Most commonly, Crohn's affects the small intestine and the beginning of the large intestine. However, the disease can affect any part of the GI tract, from the mouth to the anus.
                
Crohn's disease is a chronic inflammatory disease of the GI tract, called inflammatory bowel disease (IBD). Ulcerative colitis and microscopic colitis are the other common IBDs. More information is provided in the NIDDK health topics, Ulcerative Colitis and Microscopic Colitis: Collagenous Colitis and Lymphocytic Colitis.
                
Crohn's disease most often begins gradually and can 

## Complete the QLoRA adapter


configurando LoRA (Low-Rank Adaptation), la técnica que permite hacer fine-tuning a un LLM modificando apenas un 0.1% a 1% de sus parámetros, lo que hace posible entrenarlo en una GPU modesta en minutos.

2. ¿Qué es lora_dropout=0.05?
El Dropout apaga aleatoriamente el 5% de las neuronas de los adaptadores LoRA durante cada paso de entrenamiento.
¿Por qué? Como tu dataset de entrenamiento es pequeño (96 ejemplos), el riesgo número uno es el sobreajuste (overfitting): que el modelo memorice las respuestas exactas en lugar de aprender el estilo y razonamiento médico. Un dropout bajo como 0.05 actúa como regularizador para forzar al modelo a ser más robusto y generalizar mejor.

TODO 7
Set adapter dropout and choose which linear layers receive LoRA adapters.

Check: A dropout no greater than 0.1 and adapters applied to all linear layers.

A small dropout such as 0.05 is suitable here. PEFT accepts "all-linear" for the target modules.

In [15]:
lora = LoraConfig(
    r=8, # rank; balance memory and learning
    lora_alpha=16, # lora standard is 2*r; learned weights is multiplied by 2 determining the force from the new adaptations over base model
    # TODO 7: choose dropout and the linear layers that receive adapters.
    lora_dropout=0.05, # turn off aleatory 5% neurons from LoRA adapters on each train step
    bias="none", # we not training bias. saving memory
    target_modules="all-linear", # Dettmers et al., 2023 manifest empirically all linear is better with LoRA
    task_type="CAUSAL_LM", # indicate that is a auto regressive model
)
print(f"LoRA rank: {lora.r}")


LoRA rank: 8


## Fine-tune the model


Fine-tune the model
TODO 8
Connect the model, training settings, formatted dataset, QLoRA adapter, and tokenizer to SFTTrainer, then train once.

Check: One configured trainer and one completed call to trainer.train().

Pass base_model, training_config, train, lora, and tokenizer to the matching SFTTrainer arguments.

In [16]:
training_config = SFTConfig(
    output_dir="/tmp/qwen-medquad",
    learning_rate=0.0002, #with lora base model weights are freeze. apply larger learning rate to adapters faster learning. value consenseused by paper qlora
    num_train_epochs=2,
    per_device_train_batch_size=2, # 2 sequences, compute gradients
    gradient_accumulation_steps=4, # sum during 4 steps and the fourth step execute an unique optimizer.step(). Just use a small fraction of memory
    max_length=384, #max tokens
    logging_steps=1,
    save_strategy="no",
    report_to=[],
    bf16=True, # instead of fp16. optimized and standard by Google (Kalamkar et al., 2019: "A Study of BFLOAT16 for Deep Learning Training
    gradient_checkpointing=True, # delete activation during forward pass. On the backward pass, re caluclate on the fly. More time but save VRAM optimizing use of GPUs
    dataset_text_field="text",
    seed=42,
)

# TODO 8: create the SFTTrainer and start one training run.
trainer = SFTTrainer(
    model=base_model,
    train_dataset=train,
    peft_config=lora,
    processing_class=tokenizer,
    args=training_config,
)

train_result = trainer.train()

assert trainer is not None, "Create the trainer"
assert train_result is not None, "Start the training run"
print({"training_loss": round(train_result.training_loss, 5)})


Adding EOS to train dataset:   0%|          | 0/96 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/96 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/96 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/96 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/96 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
1,7.883445
2,6.416318
3,5.708977
4,4.705070
5,4.439521
6,3.987570
7,4.009421
8,3.654550
9,3.216932
10,3.174373


{'training_loss': 4.46225}


## Inspect where the model still fails


Inspect where the model still fails
TODO 9
Measure the score change, then sort the per-example results and keep the two weakest answers.

Check: A numeric score_change and exactly two result records ordered from lowest token F1.

Subtract the baseline token_f1 from the fine-tuned token_f1. Sort fine_tuned_result["results"] by token_f1 and take the first two rows.

In [17]:
trainer.model.eval()
fine_tuned_result = evaluate_model(trainer.model, dataset["evaluation"])

# TODO 9: calculate the score change and keep the two weakest trained-model answers.
score_change = fine_tuned_result["token_f1"] - baseline["token_f1"]
# order predicitons by token_f1 menor to mayor and take the 2 first
weakest = sorted(fine_tuned_result["results"], key=lambda x: x["token_f1"])[:2]

print(json.dumps({
    "baseline_token_f1": baseline["token_f1"],
    "fine_tuned_token_f1": fine_tuned_result["token_f1"],
    "score_change": score_change,
    "weakest_answers": weakest,
}, indent=2))


{
  "baseline_token_f1": 25.96,
  "fine_tuned_token_f1": 26.73,
  "score_change": 0.7699999999999996,
  "weakest_answers": [
    {
      "question": "What are the treatments for cystinuria ?",
      "reference": "These resources address the diagnosis or management of cystinuria:  - Genetic Testing Registry: Cystinuria  - MedlinePlus Encyclopedia: Cystinuria  - MedlinePlus Encyclopedia: Cystinuria (image)   These resources from MedlinePlus offer information about the diagnosis and management of various health conditions:  - Diagnostic Tests  - Drug Therapy  - Surgery and Rehabilitation  - Genetic Counseling   - Palliative Care",
      "prediction": "Cystinuria is a genetic disorder that causes the body to excrete excessive amounts of cystine in the urine. Cystine is an amino acid that is normally found in the body. When cystine is excreted in the urine, it can form crystals and stones in the kidneys and bladder. These stones can cause pain, infection, and other complications. Treatment 